# Sistema de Recomendación Colaborativo - Créditos Económicos con PySpark

Este Notebook presenta la construcción de un sistema de recomendación colaborativo basado en filtrado colaborativo usando el algoritmo **ALS (Alternating Least Squares)** en **PySpark**.

El objetivo es recomendar productos de la tienda **Créditos Económicos** (tecnología, electrodomésticos, hogar) a clientes utilizando sus patrones de consumo similares: *"personas con gustos parecidos también eligieron..."*.

### Requisitos del Proyecto Implementados:
1. **Catálogo Real de Productos**: 35 productos reales de Créditos Económicos con categorías (Tecnología, TV y Audio, Electrodomésticos, Climatización, Hogar), marcas, tienda y precios reales.
2. **Perfiles de Consumo**: Definición de perfiles de consumo específicos (Tecnológico/Gamer, Hogar/Familiar, Estudiante/Ahorrador, Fitness/Estilo de Vida) y asignación lógica de afinidades.
3. **Matriz de Utilidad**: Simulación de ratings lógicos (1.0 a 5.0) usando la regla matemática:
   $$Rating = Base + Afinidad\_Categoria + Afinidad\_Marca - Penalizacion\_Precio + Ruido$$
4. **Entrenamiento y Evaluación de ALS**: Tuning de hiperparámetros (`rank` y `regParam`) comparando **RMSE** y **MAE**.
5. **Recomendaciones**: Top 5 de productos recomendados para 5 clientes específicos y análisis de coherencia.
6. **Reflexión Conceptual**: Discusión sobre arranque en frío (Cold Start), sesgo de popularidad y privacidad.

---


## 1. Configuración de la Sesión de Spark y Carga de Librerías

Inicializamos la sesión local de PySpark y cargamos los módulos necesarios para el modelado, procesamiento y evaluación.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# PySpark Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, when, count
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Configurar Spark local
spark = SparkSession.builder \
    .appName("RecomendadorCreditosEconomicos") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Verificar la sesión
print(f"Versión de PySpark: {spark.version}")


## 2. Carga y Exploración del Dataset

Cargamos los archivos CSV de productos, clientes y calificaciones generados previamente.


In [ ]:
# Cargar datos en DataFrames de Spark
df_productos_spark = spark.read.csv("productos.csv", header=True, inferSchema=True)
df_calificaciones_spark = spark.read.csv("calificaciones.csv", header=True, inferSchema=True)
df_clientes_spark = spark.read.csv("clientes.csv", header=True, inferSchema=True)

# Registrar como vistas temporales para consultas SQL de ser necesario
df_productos_spark.createOrReplaceTempView("productos")
df_calificaciones_spark.createOrReplaceTempView("calificaciones")
df_clientes_spark.createOrReplaceTempView("clientes")

print("Estructura de Productos:")
df_productos_spark.show(5, truncate=False)

print("Estructura de Clientes:")
df_clientes_spark.show(5, truncate=False)

print("Estructura de Calificaciones:")
df_calificaciones_spark.show(5, truncate=False)


### Estadísticas Descriptivas del Dataset
Analizamos la cantidad de registros, distribución de calificaciones y distribución de productos por categoría.


In [ ]:
# Cantidad de datos
n_productos = df_productos_spark.count()
n_clientes = df_clientes_spark.count()
n_calificaciones = df_calificaciones_spark.count()

print(f"Total de Productos: {n_productos}")
print(f"Total de Clientes: {n_clientes}")
print(f"Total de Calificaciones: {n_calificaciones}")

# Distribución de calificaciones
print("\nEstadísticas descriptivas de los Ratings:")
df_calificaciones_spark.describe("rating").show()

# Distribución de productos por categoría
print("Productos por categoría:")
df_productos_spark.groupBy("categoria").count().show()


## 3. Matriz de Utilidad (Usuario - Producto)

La matriz de utilidad representa el interés o rating de cada usuario por cada producto. 
Visualizamos la matriz pivotada usando Pandas para obtener una mejor representación visual y calculamos la densidad de la matriz.


In [ ]:
# Convertir a pandas para pivotar y visualizar
pdf_calificaciones = df_calificaciones_spark.toPandas()
pdf_productos = df_productos_spark.toPandas()

matriz_utilidad = pdf_calificaciones.pivot(index='id_usuario', columns='id_producto', values='rating')

# Mostrar las primeras 15 filas y 15 columnas de la matriz de utilidad
print("Matriz de utilidad (fragmento de 15 usuarios x 15 productos):")
display(matriz_utilidad.iloc[:15, :15].fillna("-"))

# Calcular la densidad y el grado de escasez (sparsity)
total_celdas = n_clientes * n_productos
celdas_vacias = total_celdas - n_calificaciones
sparsity = (celdas_vacias / total_celdas) * 100
densidad = (n_calificaciones / total_celdas) * 100

print(f"\nDetalles de la matriz:")
print(f"- Total de celdas posibles: {total_celdas}")
print(f"- Celdas con calificación: {n_calificaciones}")
print(f"- Esparcimiento (Sparsity): {sparsity:.2f}% (celdas vacías)")
print(f"- Densidad de la matriz: {densidad:.2f}% (celdas calificadas)")

# Visualización de la distribución de Ratings
plt.figure(figsize=(8, 5))
sns.histplot(pdf_calificaciones['rating'], bins=15, kde=True, color='royalblue')
plt.title('Distribución de Ratings en el Dataset de Créditos Económicos')
plt.xlabel('Rating (1.0 - 5.0)')
plt.ylabel('Frecuencia')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


## 4. Entrenamiento y Optimización de ALS (Alternating Least Squares)

El algoritmo ALS de Spark requiere que los identificadores de usuarios y productos sean de tipo numérico (entero). En nuestro caso, `id_usuario` e `id_producto` ya son enteros secuenciales.

### División de Datos
Dividimos el dataset de calificaciones en:
- **80% Entrenamiento (Train)**: Usado para aprender los factores latentes.
- **20% Prueba (Test)**: Usado para validar el rendimiento y calcular métricas.

Configuramos el parámetro `coldStartStrategy="drop"` para evitar valores `NaN` en las predicciones durante la evaluación.


In [ ]:
# División aleatoria 80/20 con una semilla fija para reproducibilidad
(training_data, test_data) = df_calificaciones_spark.randomSplit([0.8, 0.2], seed=42)

print(f"Registros en Entrenamiento: {training_data.count()}")
print(f"Registros en Prueba: {test_data.count()}")


### Bucle de Hiperparámetros (Grid Search)
Evaluaremos diferentes combinaciones de los siguientes hiperparámetros:
- `rank` (Número de factores latentes): `[5, 10, 15]`
- `regParam` (Parámetro de regularización L2): `[0.01, 0.1, 0.5]`

Para cada combinación, calcularemos el **RMSE (Root Mean Squared Error)** y el **MAE (Mean Absolute Error)**.


In [ ]:
# Definir cuadrícula de hiperparámetros
ranks = [5, 10, 15]
regParams = [0.01, 0.1, 0.5]

resultados = []

# Evaluadores
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")

print("Iniciando Grid Search...")
print("-" * 50)
print(f"{'Rank':<8}{'RegParam':<12}{'RMSE':<12}{'MAE':<12}")
print("-" * 50)

for rank in ranks:
    for reg in regParams:
        # Configurar y entrenar el modelo ALS
        als = ALS(
            maxIter=15,
            regParam=reg,
            rank=rank,
            userCol="id_usuario",
            itemCol="id_producto",
            ratingCol="rating",
            coldStartStrategy="drop",
            seed=42
        )
        
        modelo = als.fit(training_data)
        
        # Predicciones sobre el conjunto de test
        predicciones = modelo.transform(test_data)
        
        # Calcular métricas
        rmse = evaluator_rmse.evaluate(predicciones)
        mae = evaluator_mae.evaluate(predicciones)
        
        resultados.append({
            "rank": rank,
            "regParam": reg,
            "rmse": rmse,
            "mae": mae
        })
        
        print(f"{rank:<8}{reg:<12}{rmse:<12.4f}{mae:<12.4f}")

df_resultados = pd.DataFrame(resultados)
print("-" * 50)


### Comparación y Selección del Mejor Modelo

Analizamos los resultados en una tabla comparativa y seleccionamos la combinación de hiperparámetros que minimiza el error RMSE.


In [ ]:
# Ordenar resultados por RMSE ascendente
df_resultados_sorted = df_resultados.sort_values(by="rmse")
print("Resultados ordenados por mejor RMSE:")
print(df_resultados_sorted)

# Obtener mejor modelo
mejor_config = df_resultados_sorted.iloc[0]
mejor_rank = int(mejor_config["rank"])
mejor_reg = mejor_config["regParam"]

print(f"\n--> El MEJOR modelo configurado es:")
print(f"    Rank (Factores Latentes): {mejor_rank}")
print(f"    regParam (Regularización): {mejor_reg}")
print(f"    RMSE en Test: {mejor_config['rmse']:.4f}")
print(f"    MAE en Test: {mejor_config['mae']:.4f}")

# Reentrenar el mejor modelo con todos los datos de entrenamiento
als_final = ALS(
    maxIter=15,
    regParam=mejor_reg,
    rank=mejor_rank,
    userCol="id_usuario",
    itemCol="id_producto",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42
)
modelo_final = als_final.fit(df_calificaciones_spark)


### Visualización Gráfica de Errores
Graficamos los efectos de `rank` y `regParam` en el RMSE del modelo.


In [ ]:
plt.figure(figsize=(10, 6))
for r in ranks:
    subset = df_resultados[df_resultados['rank'] == r]
    plt.plot(subset['regParam'], subset['rmse'], marker='o', label=f'Rank = {r}')

plt.title('Efecto de regParam y Rank en el RMSE')
plt.xlabel('Regularization Parameter (regParam)')
plt.ylabel('RMSE')
plt.xscale('log')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()


## 5. Generación de Recomendaciones Personalizadas

Utilizaremos el mejor modelo entrenado para generar el **Top 5** de productos recomendados para **5 clientes específicos**.

Para evaluar la coherencia cualitativa, analizaremos los perfiles de consumo originales de estos 5 clientes y verificaremos si las recomendaciones se alinean con sus intereses teóricos.


In [ ]:
# 1. Generar recomendaciones globales para todos los usuarios
# recommendForAllUsers(numItems) devuelve un DataFrame con: id_usuario, recommendations [id_producto, rating]
recomendaciones_globales = modelo_final.recommendForAllUsers(5)

# Seleccionar 5 usuarios específicos para inspección
# Usuarios elegidos: ID 1, 2, 3, 4, 5
usuarios_interes = [1, 2, 3, 4, 5]

df_rec_filtrado = recomendaciones_globales.filter(col("id_usuario").isin(usuarios_interes))

print("Recomendaciones crudas (formato Spark struct):")
df_rec_filtrado.show(truncate=False)


### Interpretación y Análisis Cualitativo por Cliente

Convertimos las recomendaciones a un formato legible cruzándolas con los metadatos de productos y el perfil del cliente para evaluar la coherencia.


In [ ]:
# Convertir las recomendaciones a un formato plano (explode) para hacer JOINs
from pyspark.sql.functions import explode

df_rec_explotado = df_rec_filtrado.select(
    col("id_usuario"),
    explode(col("recommendations")).alias("rec")
).select(
    col("id_usuario"),
    col("rec.id_producto").alias("id_producto"),
    col("rec.rating").alias("rating_predicho")
)

# Unir con información de productos y clientes para enriquecer la tabla
df_rec_enriquecida = df_rec_explotado \
    .join(df_productos_spark, "id_producto") \
    .join(df_clientes_spark, "id_usuario") \
    .orderBy("id_usuario", col("rating_predicho").desc())

# Convertir a Pandas para presentación amigable
pdf_rec_enriquecida = df_rec_enriquecida.toPandas()

# Mostrar recomendaciones por usuario
for u_id in usuarios_interes:
    perfil_usuario = pdf_rec_enriquecida[pdf_rec_enriquecida['id_usuario'] == u_id]['perfil'].values[0]
    print("=" * 80)
    print(f"RECOMENDACIONES PARA EL USUARIO ID: {u_id} | PERFIL: {perfil_usuario}")
    print("=" * 80)
    
    # Mostrar catálogo recomendado
    recs_user = pdf_rec_enriquecida[pdf_rec_enriquecida['id_usuario'] == u_id]
    print(recs_user[['id_producto', 'nombre', 'categoria', 'marca', 'precio', 'rating_predicho']].to_string(index=False))
    print("\nCoherencia observada:")
    
    # Análisis simple de coherencia
    if perfil_usuario == "Tecnologico_Gamer":
        print("-> Coherente: Se sugieren laptops de alta gama, consolas, o accesorios de sonido con marcas premium.")
    elif perfil_usuario == "Hogar_Familiar":
        print("-> Coherente: Se sugieren electrodomésticos grandes (línea blanca), artículos de hogar o climatización de marcas tradicionales.")
    elif perfil_usuario == "Estudiante_Ahorrador":
        print("-> Coherente: Se sugieren productos tecnológicos de costo bajo o intermedio de marcas accesibles, respetando la alta sensibilidad al precio.")
    elif perfil_usuario == "Fitness_EstilodeVida":
        print("-> Coherente: Se sugieren gadgets de salud/deporte, freidoras de aire, licuadoras u otros pequeños electrodomésticos afines.")
    print("-" * 80)


## 6. Reflexiones y Desafíos de Recomendadores en Entornos Reales

Al desplegar sistemas de recomendación basados en filtrado colaborativo como ALS en tiendas de retail reales (como Créditos Económicos), surgen varios retos clave:

### 1. El Desafío del Arranque en Frío (Cold Start)
*   **Problema**: Ocurre cuando se añade un nuevo usuario al sistema (sin historial de calificaciones) o un nuevo producto (sin calificaciones previas). ALS no puede generar factores latentes para estas entidades, fallando en recomendarles cosas o en recomendar el nuevo artículo.
*   **Estrategias de Mitigación**:
    *   *Nuevos Usuarios*: Implementar sistemas híbridos. Al registrarse, solicitar al usuario que elija categorías o marcas de interés, o mostrar los productos "más populares" / "mejor valorados" del catálogo general.
    *   *Nuevos Productos*: Usar recomendaciones basadas en contenido (Content-Based Filtering) mapeando atributos del producto (marca, precio, categoría) con los perfiles de los clientes, o promover de manera forzada nuevos productos mediante banners publicitarios (estrategia de exploración / "Epsilon-Greedy").

### 2. El Sesgo de Popularidad y Diversidad (Popularity Bias)
*   **Problema**: Los algoritmos de filtrado colaborativo tienden a recomendar de forma desproporcionada los productos que más interacciones tienen (por ejemplo, celulares Samsung o Smart TVs en oferta), ignorando la "cola larga" (Long Tail) del catálogo (como muebles específicos, colchones o climatización menor).
*   **Estrategias de Mitigación**:
    *   Introducir penalizaciones a la popularidad en la fase de recomendación final (re-ranking).
    *   Optimizar la diversidad del listado mediante algoritmos que restrinjan el número de productos de una misma categoría en la lista final.
    *   Utilizar regularizaciones específicas o entrenar con pesos inversamente proporcionales a la popularidad del artículo.

### 3. Privacidad y Uso Ético de los Datos de Clientes
*   **Problema**: Recopilar interacciones de navegación, búsquedas y compras de clientes plantea riesgos de privacidad y seguridad de la información. En Ecuador, esto se rige bajo la *Ley Orgánica de Protección de Datos Personales (LOPDP)*.
*   **Estrategias de Mitigación**:
    *   *Consentimiento Informado*: Garantizar que los usuarios acepten explícitamente el uso de sus datos para personalización.
    *   *Anonimización*: Procesar el modelado ALS utilizando únicamente IDs numéricos desvinculados de datos de carácter personal directo (nombres, cédulas, correos).
    *   *Seguridad*: Implementar encriptación en tránsito y reposo para los datasets y bases de datos que alimentan el recomendador.

---
**Desarrollado por**: [Tu Nombre / Tu Grupo]
